# Phase 0 - encode the 81 dev queries (Colab GPU)

This notebook is **self-contained**: no repository upload and no index upload.
It embeds the 40 dev gold questions + 41 dev probes and produces
`phase0_dev_query_cache.zip` in the exact `QueryVectorStore` format.

On the laptop, unzip it so the files land in `data/query_cache/`.

Runtime: a few minutes on a T4 after the model downloads. The ColQwen model is
about 9 GB in memory; use a GPU runtime.


In [ ]:
!nvidia-smi || true
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> GPU'


In [ ]:
%pip install -q colpali-engine pillow
!pip install -q -U torchao


In [ ]:
DEV_ITEMS = [{'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-0', 'question': 'How much is the Trading Operating Profit in 2011?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-1', 'question': 'How much is the Trading Operating Profit in the year Nestlé achieved the third largest Organic Growth in 10 years?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-2', 'question': 'Which achieved better performance in 2011, Organic Growth or Real Internal Growth?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-3', 'question': 'Which achieved better performance in the year Nestlé acquired Prometheus, Organic Growth or Real Internal Growth?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-4', 'question': 'In which year did Nestlé achieve higher Organic Growth, 2003 or 2004?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-5', 'question': 'In which year did the company that divested Eismann in 2004 achieve higher Organic Growth, 2003 or 2004?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-6', 'question': 'What is the name of the brand which Nestlé acquired in 2000?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-7', 'question': 'What is the name of the brand which Nestlé acquired in the year Findus was divested?'}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-8', 'question': "What is the percentage of Nestlé's 10-year annual average of the real internal growth?"}, {'qid': '2012-02-20fy11roadshow-120221022442-phpapp02_95-9', 'question': 'What is the percentage of the 10-year annual average of the Real Internal Growth for the company which has the tagline "Good Food, Good Life"?'}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-10', 'question': 'How many kinds of BICC Key Skills are there?'}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-11', 'question': 'How many kinds of Business Intelligence Competence Center Key Skills are there?'}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-12', 'question': 'What is the total percentage that wait Minutes and Hours for an average BI request?'}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-13', 'question': 'What is the biggest reason why users sometimes choose not to use enterprise BI applications?'}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-14', 'question': 'What is the biggest reason why users sometimes choose not to use the applications, data of which are not mostly trusted by 8% users?'}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-15', 'question': 'Which receives lower percentage of answers, Enterprise applications "Are too complex, complicated, cumbersome to use" or "Take too long to get results"?'}, {'qid': '0903organizingforbiandbigdatainthe21stcentury-clean-140922112024-phpapp02_95-16', 'question': 'What is the percentage of the respondents who answer "I use only enterprise BI applications"?'}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-17', 'question': 'What is the title of the presentation that ALCATEL-LUCENT made for the OVERVIEW of 2012?'}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-18', 'question': 'What is the title of the presentation that the company which had won 7 Nobel Prizes made for the OVERVIEW of 2012?'}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-19', 'question': 'Which region of the world accounts for the highest percentage of 2010 revenues?'}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-20', 'question': 'Which region of the world accounts for the highest percentage of revenues in the year 12% GROWTH is achieved?'}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-21', 'question': 'Where does Alcatel-Lucent have the IP Transformation Center in Europe?'}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-22', 'question': 'Where does the company which was awarded 2400 Patents in 2010 have the IP Transformation Center in Europe?'}, {'qid': 'alcatel-lucenteesoverview-120403090342-phpapp01_95-23', 'question': 'How many Mobile Operators are listed as References / proof of our expertise?'}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-24', 'question': 'Which Activity is undertaken by more smartphone users, Email services or Social networks?'}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-25', 'question': 'Which country spend the most in Mobile display ads in 2013, according to IAB Europe AdEx Benchmark (2013)?'}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-26', 'question': 'Which country spend the most in Mobile display ads in the year 79300 Romanian use Twitter, according to IAB Europe AdEx Benchmark (2013)?'}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-27', 'question': 'What is the total percentage of 4G users with the 1 - 500 MB plan and the 501 MB - 1 GB plan in the UK?'}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-28', 'question': 'Which country had the higher percentage of Tablet Pageviews in June 2014, Russia or Hungary?'}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-29', 'question': 'Which country had the higher percentage of Tablet Pageviews in June 2014, Russia or the country where 61 million euros was spent in online display ads in 2013?'}, {'qid': '8eeomnwqsmdmin6levsv-signature-2693c633d5e2c6d5cfd1372a7aa1d1a8df13f59c942a848ca9497686d609bddd-poli-150601151510-lva1-app6892_95-30', 'question': 'Which OS was used more among mobile Facebook users in October 2014, Android or Windows Mobile?'}, {'qid': 'a-151012171903-lva1-app6891_95-31', 'question': 'How many points are listed for TAM Notes?'}, {'qid': 'a-151012171903-lva1-app6891_95-32', 'question': 'What is the abbreviation for Sales Qualified Leads?'}, {'qid': 'a-151012171903-lva1-app6891_95-33', 'question': 'How many Partner companies are listed in the example of Go to Market Notes?'}, {'qid': 'advertisingpsychologyfinal-090705155755-phpapp02_95-34', 'question': 'What is the Major objective other than capturing attention and making a useful lasting impression?'}, {'qid': 'advertisingpsychologyfinal-090705155755-phpapp02_95-35', 'question': 'What are listed as the examples of Social characteristics in A framework of psychological meaning?'}, {'qid': 'a-3-c2b8b6654f47bdfbff1433386996d789694f921a-150414014636-conversion-gate01_95-36', 'question': 'What is a term that refers to the way work is assigned to the team?'}, {'qid': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95-37', 'question': 'In which year is the percentage of the Subnational Debts guaranteed by Governments higher, 2015 or 2016?'}, {'qid': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95-38', 'question': 'In which year may Subnational Debts be relieved more by Government, 2015 or 2016?'}, {'qid': '1shangxichinassubnationaldebts-issuesandsuggestions-160229201053_95-39', 'question': 'Which are the bigger Debts to be repaid as of June 2013, Provincial-level government debts or Municipal-level government debts?'}, {'qid': 'unans-00', 'question': 'What was the closing stock price of Alcatel-Lucent on December 31, 2013?'}, {'qid': 'unans-01', 'question': 'How many kinds of BICC Key Skills are there in 2020?'}, {'qid': 'unans-02', 'question': "What is the percentage of Nestle's 10-year annual average of the real internal growth as reported in 2021?"}, {'qid': 'unans-03', 'question': 'Which are the bigger Debts to be repaid as of March 2019, Provincial-level government debts or Municipal-level government debts?'}, {'qid': 'unans-04', 'question': 'How much is the Trading Operating Profit in 2019?'}, {'qid': 'unans-05', 'question': 'Which achieved better performance in 2020, Organic Growth or Real Internal Growth?'}, {'qid': 'unans-06', 'question': 'In which year did Nestlé achieve higher Organic Growth, 2021 or 2022?'}, {'qid': 'unans-07', 'question': 'Which region of the world accounts for the highest percentage of 2023 revenues?'}, {'qid': 'unans-08', 'question': 'In which year is the percentage of the Subnational Debts guaranteed by Governments higher, 2025 or 1999?'}, {'qid': 'unans-09', 'question': 'In which year may Subnational Debts be relieved more by Government, 1998 or 1997?'}, {'qid': 'unans-10', 'question': 'Which are the bigger Debts to be repaid as of June 1995, Provincial-level government debts or Municipal-level government debts?'}, {'qid': 'unans-11', 'question': 'Which are the bigger Debts to be repaid, Provincial-level government debts or Municipal-level government debts in June 2012?'}, {'qid': 'unans-12', 'question': 'Which country had the higher percentage of Tablet Pageviews in July 2014, Russia or Hungary?'}, {'qid': 'unans-13', 'question': "What is the percentage of Nestle's 20-year annual average of the real internal growth?"}, {'qid': 'unans-14', 'question': 'What is the total percentage that wait Minutes and Hours for an average BI request in 2020?'}, {'qid': 'unans-15', 'question': 'Which Activity is undertaken by more laptop users, Email services or Social networks?'}, {'qid': 'unans-16', 'question': 'Which OS was used more among mobile Facebook users in November 2014, Android or Windows Mobile?'}, {'qid': 'unans-17', 'question': 'Which country spend the most in Mobile display ads in 2019, according to IAB Europe AdEx Benchmark?'}, {'qid': 'unans-18', 'question': 'Which region of the world accounts for the highest percentage of 2019 revenues?'}, {'qid': 'unans-19', 'question': 'How many employees does Zephyrix Analytics have in Zurich?'}, {'qid': 'unans-20', 'question': 'What was the total revenue of Zynga in 2011?'}, {'qid': 'unans-21', 'question': 'How many 5G subscriptions were active worldwide in 2020?'}, {'qid': 'unans-22', 'question': 'How much did Nestle pay to acquire Wyeth Nutrition?'}, {'qid': 'unans-23', 'question': "What was Siemens' organic growth in 2011 according to the Nestle roadshow?"}, {'qid': 'unans-24', 'question': 'Which company acquired Alcatel-Lucent in 2016?'}, {'qid': 'unans-25', 'question': 'What was the market share of Xiaomi in the global smartphone market?'}, {'qid': 'unans-26', 'question': 'How many employees does Samsung Electronics have in Seoul?'}, {'qid': 'unans-27', 'question': 'How many users does WeChat have in Europe?'}, {'qid': 'unans-28', 'question': 'How many employees does Microsoft have worldwide?'}, {'qid': 'unans-29', 'question': "What was Sony's revenue in 2011?"}, {'qid': 'unans-30', 'question': 'How many members does LinkedIn have?'}, {'qid': 'unans-31', 'question': 'What is the estimated cost of the Qatar 2022 World Cup?'}, {'qid': 'unans-32', 'question': 'Who was the CEO of Intel who replaced Paul Otellini in 2013?'}, {'qid': 'unans-33', 'question': 'What is the second largest mobile operator by revenue mentioned in the Alcatel-Lucent references?'}, {'qid': 'unans-34', 'question': 'In the year Nestle was founded, what was its organic growth?'}, {'qid': 'unans-35', 'question': 'What is the name of the brand which Nestle acquired in the year the Berlin Wall fell?'}, {'qid': 'unans-36', 'question': 'What was the revenue of the company that acquired WhatsApp?'}, {'qid': 'unans-37', 'question': 'Who was the CEO of Nestle when the iPhone was released?'}, {'qid': 'unans-38', 'question': 'What does the Alcatel-Lucent deck say will happen after Brexit?'}, {'qid': 'unans-39', 'question': 'Which of these documents mentions the Amazon rainforest?'}, {'qid': 'unans-40', 'question': 'What was the Alcatel-Lucent stock price on the day Twitter went public?'}]


In [ ]:
import json
print('embedded questions:', len(DEV_ITEMS))
print('first:', DEV_ITEMS[0]['question'])
print('last :', DEV_ITEMS[-1]['question'])


In [ ]:
import hashlib
import json
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from colpali_engine.models import ColQwen2, ColQwen2Processor

MODEL_ID = 'vidore/colqwen2-v1.0'
OUT = Path('query_cache_phase0')
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True, exist_ok=True)

def qkey(q: str) -> str:
    return hashlib.sha1(q.strip().encode('utf-8')).hexdigest()

seen = {}
for item in DEV_ITEMS:
    q = item['question']
    seen.setdefault(qkey(q), q)
print('unique query keys:', len(seen))

processor = ColQwen2Processor.from_pretrained(MODEL_ID)
model = ColQwen2.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='cuda',
).eval()

t0 = time.time()
dims = set()
for i, (key, q) in enumerate(seen.items(), 1):
    batch = processor.process_queries([q]).to('cuda')
    with torch.no_grad():
        emb = model(**batch)
    emb = emb[0].detach().float().cpu().numpy()
    dims.add(tuple(emb.shape))
    (OUT / f'{key}.json').write_text(json.dumps({
        'question': q,
        'vectors': emb.tolist(),
        'meta': {
            'model_id': MODEL_ID,
            'prompt': 'ColQwen2Processor.process_queries(q)',
            'source': 'phase0_dev_queries_colab.ipynb',
        },
    }), encoding='utf-8')
    if i % 10 == 0 or i == len(seen):
        print(f'{i}/{len(seen)}  {time.time()-t0:.1f}s')

print('dims:', sorted(dims))
assert len(list(OUT.glob('*.json'))) == len(seen)


In [ ]:
import shutil
from pathlib import Path

zip_path = Path('phase0_dev_query_cache.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir='.', base_dir='query_cache_phase0')
print('wrote', zip_path, zip_path.stat().st_size / 1e6, 'MB')

from google.colab import files
files.download(str(zip_path))


## Laptop handoff

```bash
# from the repo root
unzip phase0_dev_query_cache.zip -d data/query_cache
python - <<'PY'
from pathlib import Path
print(len(list(Path('data/query_cache').glob('*.json'))), 'cached queries')
PY
```

Then run the sweep notebook or the local sweep command. The cache holds 81
vectors; the dev gold/probe question hashes must match exactly.
